In [ ]:
#%pip install --upgrade xgboost
#%pip install --upgrade pip

In [ ]:
import copy
import json
import random
import time
from pathlib import Path
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
import joblib

In [ ]:
FEATURES = ['S0', 'K', 'T', 'r', 'v0', 'kappa', 'theta', 'sigma', 'rho']
TRAIN_N, VALIDATION_N, TEST_N = 12000, 200, 500
EPOCHS, PATIENCE, BATCH_SIZE, LEARNING_RATE, MIN_DELTA = 1000, 100, 64, 1e-3, 1e-7
MODEL_SEED = (42, 43, 44)
HIDDEN_UNITS = (128, 128, 128)

REUSE_BASELINE = False
BASELINE_RESULTS_DIR = Path('experiment_baseline')
DATA_DIR = Path('experiment_training_size_results') / 'data'
ARCHITECTURE_DIR = Path('experiment_architecture_results')/f'n{TRAIN_N}'
OUTPUT_DIR = Path('experiment_model_comparison_results') / f'n{TRAIN_N}'
MODEL_DIR = OUTPUT_DIR / 'models'
HISTORY_DIR = OUTPUT_DIR / 'histories'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
FIGURE_DIR = OUTPUT_DIR / 'figures'

for directory in (MODEL_DIR, HISTORY_DIR, PREDICTION_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cpu')
#torch.set_num_threads(1)

XGB_PARAMS = dict(
    n_estimators=100,
    max_depth=5,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    #n_jobs=1
)
print('Training Size: ', TRAIN_N)


Training Size:  12000


In [ ]:
def check_data(df, name):
    required = FEATURES + ['price']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{name}: Missing columns: {missing}')
    if not np.isfinite(df[required].to_numpy(dtype=float)).all():
        raise ValueError(f'{name}: Non-finite data detected')
    if (df['price'] < -1e-7).any():
        raise ValueError(f'{name}: Materially negative reference prices detected')
    lower = np.maximum(df['S0'] - df['K'] * np.exp(-df['r'] * df['T']),0.0)
    if (df['price'] < lower - 1e-4).any() or (df['price'] > df['S0'] + 1e-4).any():
        raise ValueError(f'{name}: Option prices violate no-arbitrage bounds')
    if df[FEATURES].duplicated().any():
        raise ValueError(f"{name}:Duplicate parameter combinations in a dataset")
base_train = pd.read_csv(BASELINE_RESULTS_DIR/'train.csv')
extension = pd.read_csv(DATA_DIR/'extension_11000.csv')
validation = pd.read_csv(BASELINE_RESULTS_DIR/'validation.csv')
test = pd.read_csv(BASELINE_RESULTS_DIR/'test.csv')

train_pool = pd.concat([base_train, extension], ignore_index=True)
indices = pd.read_csv(DATA_DIR/f'training_indices_{TRAIN_N}.csv')['training_pool_row_index']
if not np.isfinite(indices.to_numpy(dtype=float)).all() or not (indices % 1 == 0).all():
    raise ValueError('Training indices must be finite integers')
assert len(base_train) == 1000 and len(extension) == 11000
assert len(indices) == TRAIN_N and indices.is_unique
assert (indices >= 0).all() and (indices < len(train_pool)).all()
train = train_pool.iloc[indices.to_numpy(dtype=int)].copy().reset_index(drop=True)

for frame, name in [(train_pool, 'training_pool'), (train, 'train'), (validation, 'validation'), (test, 'test')]:
    check_data(frame, name)
assert len(validation) == VALIDATION_N and len(test) == TEST_N
assert not pd.concat([train_pool, validation, test])[FEATURES].duplicated().any()

x_scaler = StandardScaler().fit(train[FEATURES].to_numpy())
y_scaler = StandardScaler().fit(train[['price']].to_numpy())

print(f'Train: {len(train):,} | Validation: {len(validation)} | Test: {len(test)}')

X_train = train[FEATURES].to_numpy()
X_val = validation[FEATURES].to_numpy()
X_test = test[FEATURES].to_numpy()
y_train = train['price'].to_numpy()
y_val = validation['price'].to_numpy()
y_test = test['price'].to_numpy()

xtr = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
xva = torch.tensor(x_scaler.transform(X_val), dtype=torch.float32)
xte = torch.tensor(x_scaler.transform(X_test), dtype=torch.float32)

Train: 12,000 | Validation: 200 | Test: 500


In [ ]:
def black_scholes_price(frame):
    S0 = frame['S0'].to_numpy()
    K = frame['K'].to_numpy()
    T = frame['T'].to_numpy()
    r = frame['r'].to_numpy()
    v0 = frame['v0'].to_numpy()
    kappa = frame['kappa'].to_numpy()
    theta = frame['theta'].to_numpy()

    a = kappa * T
    weight = np.divide(-np.expm1(-a), a, out=np.ones_like(a), where=a != 0)
    variance = theta + (v0 - theta) * weight
    vol = np.sqrt(variance)

    d1 = (np.log(S0 / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

bs_train = black_scholes_price(train)
bs_val = black_scholes_price(validation)
bs_test = black_scholes_price(test)
residual_train = y_train - bs_train
residual_val = y_val - bs_val

print('Mean absolute BS residual (training):', np.mean(np.abs(residual_train)))

Mean absolute BS residual (training): 0.0026213149669423695


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_units):
        super().__init__()
        layers = []
        width = len(FEATURES)
        for hidden_width in hidden_units:
            layers.extend([
                nn.Linear(width, hidden_width),
                nn.ReLU()])
            width = hidden_width
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

def metrics(actual, pred):
    actual = np.asarray(actual, dtype=float).ravel()
    pred = np.asarray(pred, dtype=float).ravel()
    if actual.size == 0 or actual.shape != pred.shape:
        raise ValueError(
            'Actual and predicted prices must have matching non-empty shapes'
        )

    if not np.isfinite(actual).all() or not np.isfinite(pred).all():
        raise ValueError('Non-finite prices in evaluation')

    error = pred - actual
    absolute = np.abs(error)
    valid = actual >= 0.01
    return {
        'mae': float(np.mean(absolute)),
        'rmse': float(np.sqrt(np.mean(error**2))),
        'median_absolute_error': float(np.median(absolute)),
        'p90_absolute_error': float(np.percentile(absolute, 90)),
        'max_absolute_error': float(np.max(absolute)),
        'relative_error_percent_on_prices_above_0_01': (
            float(100 * np.mean(absolute[valid]/actual[valid]))
            if np.any(valid) else None
        ),
    }

In [ ]:
def fit_mlp(target_train, target_val, name, model_seed):
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    target_scaler = StandardScaler().fit(target_train.reshape(-1,1))
    ytr = torch.tensor(target_scaler.transform(target_train.reshape(-1,1)), dtype=torch.float32)
    yva = torch.tensor(target_scaler.transform(target_val.reshape(-1,1)), dtype=torch.float32)

    net = MLP(HIDDEN_UNITS).to(DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    best_state, best_epoch = None, None
    stale = 0
    history = []
    start = time.perf_counter()


    for epoch in range(1, EPOCHS+1):
        net.train()
        perm = torch.randperm(len(xtr))
        for indices in perm.split(BATCH_SIZE):
            optimizer.zero_grad()
            loss = criterion(net(xtr[indices]), ytr[indices])
            loss.backward()
            optimizer.step()
        net.eval()
        with torch.inference_mode():
            train_loss = criterion(net(xtr), ytr).item()
            val_loss = criterion(net(xva), yva).item()
        history.append({
            'epoch': epoch,
            'train_mse_scaled': train_loss,
            'validation_mse_scaled': val_loss
        })
        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise RuntimeError(f'Non-finite loss: {name}, seed={model_seed}')
        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(net.state_dict())
            stale=0
        else:
            stale += 1
        if epoch % 100 == 0:
            print(f'{name} seed={model_seed} epoch={epoch:4d} validation MSE = {val_loss:.6f}')
        if stale >= PATIENCE:
            break
    if best_state is None:
        raise RuntimeError('No valid checkpoint was obtained')
    net.load_state_dict(best_state)
    net.eval()
    stem = f'{name}_seed{model_seed}'
    pd.DataFrame(history).to_csv(HISTORY_DIR / f'history_{stem}.csv', index=False)
    torch.save({
        'model_state_dict': net.state_dict(),
        'features': FEATURES,
        'hidden_units': HIDDEN_UNITS,
        'x_mean': x_scaler.mean_,
        'x_scale': x_scaler.scale_,
        'y_mean': target_scaler.mean_,
        'y_scale': target_scaler.scale_,
        'model_seed': model_seed,
        'train_n': TRAIN_N,
        'target': 'residual' if name == 'Hybrid' else 'price',
        'bs_volatility': 'expected_average_variance' if name == 'Hybrid' else None
    },
        MODEL_DIR/f'model_{stem}.pt')
    print(f'{name} seed={model_seed}: best epoch={best_epoch}, '
          f'training={time.perf_counter() - start:.2f}s')
    return net, target_scaler

def mlp_predict(net, scaler, x):
    net.eval()
    with torch.inference_mode():
        return scaler.inverse_transform(
            net(x).cpu().numpy()
        ).ravel()

In [ ]:
results = []

def evaluate_model(name, model_seed, pred_val, pred_test):
    pred_val = np.asarray(pred_val).ravel()
    pred_test = np.asarray(pred_test).ravel()
    results.append({'model': name, 'model_seed': model_seed,
                    **{f'validation_{k}': v for k, v in metrics(y_val, pred_val).items()},
                    **metrics(y_test, pred_test)})
    predictions = test.copy()
    predictions['predicted_price'] = pred_test
    predictions['absolute_error'] = np.abs(pred_test - y_test)
    stem = name.replace(' ', '_') + f'_seed{model_seed}'
    predictions.to_csv(PREDICTION_DIR / f'test_prediction_{stem}.csv', index=False)
    print(f'{name} seed={model_seed}: validation MAE={metrics(y_val, pred_val)["mae"]:.6f}')

In [ ]:
model = DummyRegressor(strategy='mean')
model.fit(X_train, y_train)
evaluate_model('Dummy Regressor', 42, model.predict(X_val), model.predict(X_test))
joblib.dump(model, MODEL_DIR / 'Dummy_Regression.joblib')

Dummy Regressor seed=42: validation MAE=0.125309


['experiment_model_comparison_results/n12000/models/Dummy_Regression.joblib']

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
evaluate_model('Linear Regression', 42, model.predict(X_val), model.predict(X_test))
joblib.dump(model, MODEL_DIR / 'Linear_Regression.joblib')

Linear Regression seed=42: validation MAE=0.024622


['experiment_model_comparison_results/n12000/models/Linear_Regression.joblib']

In [ ]:
evaluate_model('Black-Scholes', 42, bs_val, bs_test)

Black-Scholes seed=42: validation MAE=0.002648


In [ ]:
from xgboost import XGBRegressor
for model_seed in MODEL_SEED:
    print(f'Training XGBoost seed={model_seed}', flush=True)
    print('Creating model...', flush=True)
    model = XGBRegressor(**XGB_PARAMS, random_state=model_seed)
    print('Fitting model...', flush=True)
    model.fit(X_train, y_train)
    print('Predicting...', flush=True)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)

    print('Evaluating...', flush=True)
    evaluate_model('XGBoost', model_seed, model.predict(X_val), model.predict(X_test))
    print('Saving', flush=True)
    model.save_model(str(MODEL_DIR/f'XGBoost_seed{model_seed}.json'))

Training XGBoost seed=42
Creating model...
Fitting model...
Predicting...
Evaluating...
XGBoost seed=42: validation MAE=0.008390
Saving
Training XGBoost seed=43
Creating model...
Fitting model...
Predicting...
Evaluating...
XGBoost seed=43: validation MAE=0.008094
Saving
Training XGBoost seed=44
Creating model...
Fitting model...
Predicting...
Evaluating...
XGBoost seed=44: validation MAE=0.007974
Saving


In [ ]:
def restore_baseline(model_seed):
    path = ARCHITECTURE_DIR / 'models' / f'model_Baseline_seed{model_seed}.pt'
    if not path.exists():
        raise FileNotFoundError(f'{path} missing; set REUSE_BASELINE = False to retrain')
    import inspect
    kwargs = {'map_location': DEVICE}
    if 'weights_only' in inspect.signature(torch.load).parameters:
        kwargs['weights_only'] = False
    checkpoint = torch.load(path, **kwargs)
    if (list(checkpoint['features']) != FEATURES or
        tuple(checkpoint['hidden_units']) != HIDDEN_UNITS or
        checkpoint['model_seed'] != model_seed
        or checkpoint['train_n'] != TRAIN_N):
        raise ValueError('Baseline checkpoint config. mismatch')
    for key, expected in [('x_mean', x_scaler.mean_), ('x_scale', x_scaler.scale_),
                          ('y_mean', y_scaler.mean_), ('y_scale', y_scaler.scale_)]:
        if not np.allclose(checkpoint[key], expected, rtol=1e-9, atol=1e-12):
            raise ValueError(f'Baseline checkpoint scaler mismatch:{key}')
    net = MLP(HIDDEN_UNITS).to(DEVICE)
    net.load_state_dict(checkpoint['model_state_dict'])
    return net, y_scaler

for model_seed in MODEL_SEED:
    if REUSE_BASELINE:
        net, scaler = restore_baseline(model_seed)
    else:
        net, scaler = fit_mlp(y_train, y_val, 'Baseline', model_seed)
    evaluate_model('MLP Baseline', model_seed,
                   mlp_predict(net, scaler, xva), mlp_predict(net, scaler, xte))

FileNotFoundError: experiment_architecture_results/n12000/models/model_Baseline_seed42.pt missing; set REUSE_BASELINE = False to retrain

In [ ]:
for model_seed in MODEL_SEED:
    net, residual_scaler = fit_mlp(residual_train, residual_val, 'Hybrid', model_seed)
    pred_val = bs_val + mlp_predict(net, residual_scaler, xva)
    pred_test = bs_test + mlp_predict(net, residual_scaler, xte)
    evaluate_model('Hybrid BS-MLP', model_seed, pred_val, pred_test)

In [ ]:
trials = pd.DataFrame(results)
trials.to_csv(OUTPUT_DIR / 'model_comparison_trials.csv', index=False)
metric_cols = [c for c in trials.columns if c not in ('model', 'model_seed')]
summary = trials.groupby('model', as_index=False).agg(
    repeats=('model_seed', 'count'),
    **{f'{m}_mean': (m,'mean') for m in metric_cols},
    **{f'{m}_std': (m, 'std') for m in metric_cols}
).sort_values('validation_mae_mean').reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / 'model_comparison_summary.csv', index=False)
print(summary[[
    'model',
    'repeats',
    'validation_mae_mean',
    'mae_mean',
    'mae_std',
    'rmse_mean',
    'p90_absolute_error_mean',
    'max_absolute_error_mean'
]].to_string(index=False))
print('Select by mean validation MAE: ', summary.iloc[0]['model'])

(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps({
    'experiment': 4,
    'research_question': 'RQ4',
    'features': FEATURES,
    'train_n': TRAIN_N,
    'validation_n': VALIDATION_N,
    'test_n': TEST_N,
    'model_seeds': MODEL_SEED,
    'hidden_units': HIDDEN_UNITS,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
    'learning_rate': LEARNING_RATE,
    'xgboost_params': XGB_PARAMS,
    'reused_baseline': REUSE_BASELINE,
    'bs_volatility': 'sqrt(expected time-averaged variance)',
    'model_selection': 'mean validation MAE'
}, indent=2))

In [ ]:
for metric, label in [('mae', 'MAE'), ('rmse', 'RMSE')]:
    plot_data = summary.sort_values('validation_mae_mean')
    fig, ax = plt.subplots(figsize=(9,5))
    ax.bar(plot_data['model'], plot_data[f'{metric}_mean'],
           yerr=plot_data[f'{metric}_std'].fillna(0), capsize=4)
    ax.set_yscale('log')
    ax.set_xlabel('Model')
    ax.set_ylabel(f'Test {label} (option price units, log scale)')
    ax.tick_params(axis='x', rotation=25)
    ax.set_title(f'Model comparison: {label}')
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'model_comparison_{metric}.png', dpi=220)
    plt.show()
    plt.close(fig)

print(f'Files saved to {OUTPUT_DIR.resolve()}')